In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

BASE = '/content/drive/MyDrive/customer-segmentation'
RAW  = f'{BASE}/data/raw'

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

print("Libraries loaded ✓")

Libraries loaded ✓


In [3]:
df = pd.read_excel(f'{RAW}/online_retail_II.xlsx', sheet_name='Year 2010-2011')

print(f"Shape: {df.shape}")
print(f"Rows: {df.shape[0]:,} | Columns: {df.shape[1]}")

Shape: (541910, 8)
Rows: 541,910 | Columns: 8


In [4]:
df.head(10)

,Invoice,StockCode,Description,Quantity,InvoiceDate,Price,Customer ID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.00,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.00,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.00,United Kingdom
5,536365,22752,SET 7 BABUSHKA NESTING BOXES,2,2010-12-01 08:26:00,7.65,17850.00,United Kingdom
6,536365,21730,GLASS STAR FROSTED T-LIGHT HOLDER,6,2010-12-01 08:26:00,4.25,17850.00,United Kingdom
7,536366,22633,HAND WARMER UNION JACK,6,2010-12-01 08:28:00,1.85,17850.00,United Kingdom
8,536366,22632,HAND WARMER RED POLKA DOT,6,2010-12-01 08:28:00,1.85,17850.00,United Kingdom
9,536368,22960,JAM MAKING SET WITH JARS,6,2010-12-01 08:34:00,4.25,13047.00,United Kingdom


In [5]:
print("=== DATA TYPES ===")
print(df.dtypes)

print("\n=== MISSING VALUES ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
print(pd.concat([missing, missing_pct], axis=1, keys=['Count', 'Percent']))

=== DATA TYPES ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[ns]
Price                 float64
Customer ID           float64
Country                object
dtype: object

=== MISSING VALUES ===
              Count  Percent
Invoice           0     0.00
StockCode         0     0.00
Description    1454     0.27
Quantity          0     0.00
InvoiceDate       0     0.00
Price             0     0.00
Customer ID  135080    24.93
Country           0     0.00


In [6]:
null_customers = df[df['Customer ID'].isnull()]
print(f"Rows without Customer ID: {len(null_customers):,}")
print(f"Percentage of total data: {len(null_customers)/len(df)*100:.1f}%")

print(f"\nSample of anonymous transactions:")
print(null_customers[['Invoice', 'Description', 'Quantity', 'Price', 'Country']].head())

Rows without Customer ID: 135,080
Percentage of total data: 24.9%

Sample of anonymous transactions:
     Invoice                      Description  Quantity  Price         Country
622   536414                              NaN        56   0.00  United Kingdom
1443  536544  DECORATIVE ROSE BATHROOM BOTTLE         1   2.51  United Kingdom
1444  536544  DECORATIVE CATS BATHROOM BOTTLE         2   2.51  United Kingdom
1445  536544               POLKADOT RAIN HAT          4   0.85  United Kingdom
1446  536544            RAIN PONCHO RETROSPOT         2   1.66  United Kingdom


In [10]:
dupes = df.duplicated().sum()
print(f"Duplicate rows: {dupes:,}")

print("\nSample duplicate rows:")
print(df[df.duplicated(keep=False)].sort_values('Customer ID').head(6))

Duplicate rows: 5,268

Sample duplicate rows:
       Invoice StockCode                        Description  Quantity  \
395442  571034     23494   VINTAGE DOILY DELUXE SEWING KIT          3   
395443  571034     23494   VINTAGE DOILY DELUXE SEWING KIT          3   
395455  571034     23245         SET OF 3 REGENCY CAKE TINS         4   
395388  571034     23239  SET OF 4 KNICK KNACK TINS POPPIES         6   
395410  571034     23239  SET OF 4 KNICK KNACK TINS POPPIES         6   
395371  571034     23245         SET OF 3 REGENCY CAKE TINS         4   

               InvoiceDate  Price  Customer ID Country  
395442 2011-10-13 12:47:00   5.95     12359.00  Cyprus  
395443 2011-10-13 12:47:00   5.95     12359.00  Cyprus  
395455 2011-10-13 12:47:00   4.95     12359.00  Cyprus  
395388 2011-10-13 12:47:00   4.15     12359.00  Cyprus  
395410 2011-10-13 12:47:00   4.15     12359.00  Cyprus  
395371 2011-10-13 12:47:00   4.95     12359.00  Cyprus  


In [8]:
print("=== QUANTITY STATS ===")
print(df['Quantity'].describe())

print("\n=== PRICE STATS ===")
print(df['Price'].describe())

print(f"\nNegative quantities (returns): {(df['Quantity'] < 0).sum():,}")
print(f"Zero price rows: {(df['Price'] == 0).sum():,}")
print(f"Negative price rows: {(df['Price'] < 0).sum():,}")

=== QUANTITY STATS ===
count   541910.00
mean         9.55
std        218.08
min     -80995.00
25%          1.00
50%          3.00
75%         10.00
max      80995.00
Name: Quantity, dtype: float64

=== PRICE STATS ===
count   541910.00
mean         4.61
std         96.76
min     -11062.06
25%          1.25
50%          2.08
75%          4.13
max      38970.00
Name: Price, dtype: float64

Negative quantities (returns): 10,624
Zero price rows: 2,515
Negative price rows: 2


In [9]:
cancelled = df[df['Invoice'].astype(str).str.startswith('C')]
print(f"Cancelled transactions: {len(cancelled):,}")
print(f"\nSample cancellations:")
print(cancelled[['Invoice', 'Quantity', 'Price']].head())

Cancelled transactions: 9,288

Sample cancellations:
     Invoice  Quantity  Price
141  C536379        -1  27.50
154  C536383        -1   4.65
235  C536391       -12   1.65
236  C536391       -24   0.29
237  C536391       -24   0.29


In [ ]:
# ================================================
# DATA QUALITY REPORT — Findings
# ================================================
#
# 1. Missing Customer ID: 135,080 rows (24.9%)
#    Fix: drop rows where Customer ID is null
#
# 2. Duplicate rows: 5,268
#    Fix: drop_duplicates()
#
# 3. Negative quantities: 10,624 (returns/cancellations)
#    Fix: keep only Quantity > 0
#
# 4. Cancelled invoices: 9,288 (start with 'C')
#    Fix: remove invoices starting with 'C'
#
# 5. Zero or negative price: 2,517 rows
#    Fix: keep only Price > 0
#
# 6. Missing descriptions: 1,454
#    Fix: drop these rows
# ================================================